## 0.Rdkit验证SMILES手性

In [ ]:
'''
功能:rdkit验证手性立体化学
ligand1:https://www.ebi.ac.uk/chebi/CHEBI:62456
老师给的(手性有问题): C/C(C)=C/CC[C@@H](C)C1CC[C@@]2(C)C3CC=C4C(C)(C)[C@@H](O)CCC4[C@]3(C)CC[C@@]21C
3D:结构：https://pubchem.ncbi.nlm.nih.gov/compound/Cucurbitadienol#section=3D-Conformer

ligand2(P450环氧):https://www.ebi.ac.uk/chebi/CHEBI:229949
老师给的(手性有问题): C[C@H](CCC1C(C)(C)O1)C2CC[C@@]3(C)C4CC=C5C(C)(C)[C@@H](O)CCC5[C@]4(C)CC[C@@]32C
3D:结构：https://pubchem.ncbi.nlm.nih.gov/compound/171037431#section=3D-Conformer

ligand1 = "[H][C@@]12CC=C3C(C)(C)[C@@H](O)CC[C@@]3([H])[C@]1(C)CC[C@@]1(C)[C@@]2(C)CC[C@]1([H])[C@H](C)CCC=C(C)C"
ligand2 = "[H][C@@]12CC=C3C(C)(C)[C@@H](O)CC[C@@]3([H])[C@]1(C)CC[C@@]1(C)[C@@]2(C)CC[C@]1([H])[C@H](C)CC[C@@H]1OC1(C)C"
'''
from rdkit import Chem
from rdkit.Chem import AllChem

ligand1_teacher = "C/C(C)=C/CC[C@@H](C)C1CC[C@@]2(C)C3CC=C4C(C)(C)[C@@H](O)CCC4[C@]3(C)CC[C@@]21C"
ligand1_chebi = "[H][C@@]12CC=C3C(C)(C)[C@@H](O)CC[C@@]3([H])[C@]1(C)CC[C@@]1(C)[C@@]2(C)CC[C@]1([H])[C@H](C)CCC=C(C)C"
ligand2_teacher = "C[C@H](CCC1C(C)(C)O1)C2CC[C@@]3(C)C4CC=C5C(C)(C)[C@@H](O)CCC5[C@]4(C)CC[C@@]32C"
ligand2_chebi   = "[H][C@@]12CC=C3C(C)(C)[C@@H](O)CC[C@@]3([H])[C@]1(C)CC[C@@]1(C)[C@@]2(C)CC[C@]1([H])[C@H](C)CC[C@@H]1OC1(C)C"
ligand3_chebi = "C[C@H](CC[C@H](C(C)(C)O)O)[C@H]1CC[C@@]2([C@@]1(CC[C@@]3([C@H]2CC=C4[C@H]3CC[C@@H](C4(C)C)O)C)C)C"

for name, smi in [("teacher", ligand2_teacher), ("chebi", ligand3_chebi)]:
    mol = Chem.MolFromSmiles(smi)
    Chem.AssignStereochemistry(mol, cleanIt=True, force=True, flagPossibleStereoCenters=True)
    centers = Chem.FindMolChiralCenters(mol, includeUnassigned=True, useLegacyImplementation=False)
    print(name)
    print("分子式:", Chem.rdMolDescriptors.CalcMolFormula(mol))
    print("规范SMILES:", Chem.MolToSmiles(mol))
    print("手性中心 (原子序号, R/S 或 ?未指定):", centers)
    print()

teacher
分子式: C30H50O2
规范SMILES: C[C@H](CCC1OC1(C)C)C1CC[C@@]2(C)C3CC=C4C(CC[C@H](O)C4(C)C)[C@]3(C)CC[C@]12C
手性中心 (原子序号, R/S 或 ?未指定): [(1, 'R'), (4, '?'), (9, '?'), (12, 'S'), (14, '?'), (21, 'S'), (25, '?'), (26, 'R'), (30, 'R')]

chebi
分子式: C30H52O3
规范SMILES: C[C@H](CC[C@@H](O)C(C)(C)O)[C@H]1CC[C@@]2(C)[C@@H]3CC=C4[C@@H](CC[C@H](O)C4(C)C)[C@]3(C)CC[C@]12C
手性中心 (原子序号, R/S 或 ?未指定): [(1, 'R'), (4, 'R'), (10, 'R'), (13, 'S'), (14, 'R'), (17, 'R'), (18, 'R'), (22, 'S'), (25, 'S')]



## 1. 如果没找到官方的sdf就用这个

In [ ]:
'''
功能：SMILES转sdf
'''
from rdkit import Chem
from rdkit.Chem import AllChem
smi_chebi  = "[H][C@@]12CC=C3C(C)(C)[C@@H](O)CC[C@@]3([H])[C@]1(C)CC[C@@]1(C)[C@@]2(C)CC[C@]1([H])[C@H](C)CC[C@@H]1OC1(C)C"
for smiles, name in [(ligand2_chebi, "ligand1")]:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        print(f"{name}: SMILES解析失败，跳过")
        break

    # 检查手性完整性，未指定的中心先显示？
    Chem.AssignStereochemistry(mol, cleanIt=True, force=True, flagPossibleStereoCenters=True)
    centers = Chem.FindMolChiralCenters(mol, includeUnassigned=True, useLegacyImplementation=False)
    unassigned = [c for c in centers if c[1] == '?']
    if unassigned:
        print(f"{name}存在未指定手性中心: {unassigned}，构象将由算法任意指定，需核实SMILES")

    mol.SetProp("_Name", name)
    mol = Chem.AddHs(mol)

    params = AllChem.ETKDGv3()
    params.randomSeed = 42
    params.enforceChirality = True

    cids = AllChem.EmbedMultipleConfs(mol, numConfs=20, params=params)
    if len(cids) == 0:
        print(f"{name}: 标准ETKDGv3生成失败，尝试随机坐标初始化")
        params.useRandomCoords = True
        cids = AllChem.EmbedMultipleConfs(mol, numConfs=20, params=params)
        if len(cids) == 0:
            print(f"{name}: 3D构象生成彻底失败，跳过该分子")
            break

    # 力场优化，选能量最低的构象
    energies = []
    for cid in cids:
        try:
            ff = AllChem.MMFFGetMoleculeForceField(
                mol, AllChem.MMFFGetMoleculeProperties(mol), confId=cid
            )
            ff.Minimize(maxIts=2000)
            energies.append((cid, ff.CalcEnergy()))
        except Exception as e:
            print(f"{name}: 构象{cid}优化报错: {e}")

    if not energies:
        print(f"{name}: 所有构象优化失败，跳过")
        break

    best_cid = min(energies, key=lambda x: x[1])[0]
    print(f"{name}: {len(centers)}个手性中心--->{centers}，最优构象能量 = {min(e for _, e in energies):.2f} kcal/mol")

    writer = Chem.SDWriter(f"{name}_input.sdf")
    writer.write(mol, confId=best_cid)
    writer.close()
    print(f"已生成: {name}_input.sdf\n")

ligand1: 9个手性中心 → [(0, 'R'), (7, 'S'), (11, 'S'), (12, 'R'), (16, 'R'), (18, 'S'), (22, 'R'), (23, 'R'), (27, 'S')]，最优构象能量 = 146.39 kcal/mol
已生成: ligand1_input.sdf



## 2. 突变点位确定

In [ ]:
# ligand2_5A
list1 = "PHE33,PRO34,ASP101,TRP102,ILE105,VAL126,TYR150,MET151,PHE154,LEU172,THR175,SER176,ASN179,ALA183,PRO185,LEU187,PHE193,ILE196,TYR230,LEU261,THR262,HIS295,PHE296".split(',')
# ligand3_5A
list2 = "PHE33,PRO34,ASP101,TRP102,ILE105,SER125,VAL126,TYR150,MET151,THR175,SER176,ASN179,SER181,ALA183,PRO185,CYS186,LEU187,PHE193,ILE196,TYR230,LEU261,THR262,PHE265,HIS295,PHE296".split(',')
# caver
list3 = "VAL126,ILE105,HIS100,ALA104,SER125,THR262".split(',')

# 关键motif和催化残基
critical_residues = "ASP101,HIS295,ASP260,TYR150,TYR230,HIS31,GLY32,PHE33,PRO34,TRP102".split(',')

# 将列表转换为集合
set1 = set(list1)
set2 = set(list2)
set3 = set(list3)
set_critical = set(critical_residues)

# 求前三项的并集
union_set = set1 | set2 | set3

# 剔除关键残基求差集(A中去掉B)
final_set = union_set - set_critical

# 按氨基酸后面的数字序号进行排序
final_sorted = sorted(list(final_set), key=lambda x: int(x[3:]))

print("最终需要突变的残基列表:")
print(",".join(final_sorted))
print(f"共计: {len(final_sorted)}个残基")

最终需要突变的残基列表:
HIS100,ALA104,ILE105,SER125,VAL126,MET151,PHE154,LEU172,THR175,SER176,ASN179,SER181,ALA183,PRO185,CYS186,LEU187,PHE193,ILE196,LEU261,THR262,PHE265,PHE296

共计: 22个残基


## 3. 生成饱和突变序列列表共foldx使用

In [ ]:
'''
功能:对残基名字对应改成单个字母
'''
import re

# 把PyMOL打印出来的那一串直接粘贴在引号里
pymol_output = "HIS100,ALA104,ILE105,SER125,VAL126,MET151,PHE154,LEU172,THR175,SER176,ASN179,SER181,ALA183,PRO185,CYS186,LEU187,PHE193,ILE196,LEU261,THR262,PHE265,PHE296"
# 氨基酸3字母转1字母字典
aa_map = {
    'ALA':'A', 'CYS':'C', 'ASP':'D', 'GLU':'E', 'PHE':'F', 'GLY':'G', 'HIS':'H', 
    'ILE':'I', 'LYS':'K', 'LEU':'L', 'MET':'M', 'ASN':'N', 'PRO':'P', 'GLN':'Q', 
    'ARG':'R', 'SER':'S', 'THR':'T', 'VAL':'V', 'TRP':'W', 'TYR':'Y'
}

# 清理逗号，分割字符串
raw_list = re.split(r'[,\s]+', pymol_output.strip())
# 去除空项
raw_list = [x for x in raw_list if x]
print(f"要突变的位点总数{len(raw_list)}")

residues = []
aas = []
for item in raw_list:
    # 提取字母部分PHE
    aa_3 = "".join(filter(str.isalpha, item)).upper()
    # 提取数字部分105
    res_num = "".join(filter(str.isdigit, item))
    if aa_3 in aa_map and res_num:
        residues.append(int(res_num))
        aas.append(aa_map[aa_3])
    else:
        print(f"无法识别的项 '{item}'，已跳过")

# 输出可以直接复制的结果
print(f"residues_to_mutate = {residues}")
print(f"original_aas = {aas}")

'''
2. 功能:对所有残基进行单点饱和突变,并写成标准格式到individual_list.txt文件中
'''
# 填入22个残基的编号
residues_to_mutate = residues

# 填入这22个残基原本的氨基酸单字母必须和编号一一对应
# 举例，F对应105，L对应108
original_aas = aas

# 这里的链ID通常是A，如果蛋白是B链，就改成'B'
chain_id = 'A'

# 目标：20种标准氨基酸
all_aas = ['A', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 
           'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'V', 'W', 'Y']

with open("individual_list.txt", "w") as f:
    for i in range(len(residues_to_mutate)):
        res_num = residues_to_mutate[i]
        wt_aa = original_aas[i]
        
        for mutant_aa in all_aas:
            # 跳过突变成自己的
            if mutant_aa == wt_aa: continue 
            # FoldX格式：原AA + 链 + 编号 + 新AA + 分号
            # F=原氨基酸单字母，A=链ID，105=编号，A=突变后氨基酸，分号结尾
            line = f"{wt_aa}{chain_id}{res_num}{mutant_aa};\n"
            f.write(line)
total_mutations = len(residues_to_mutate) * 19
print(f"共生成{total_mutations}个单点突变任务")
print("已生成individual_list.txt")

要突变的位点总数22
residues_to_mutate = [100, 104, 105, 125, 126, 151, 154, 172, 175, 176, 179, 181, 183, 185, 186, 187, 193, 196, 261, 262, 265, 296]
original_aas = ['H', 'A', 'I', 'S', 'V', 'M', 'F', 'L', 'T', 'S', 'N', 'S', 'A', 'P', 'C', 'L', 'F', 'I', 'L', 'T', 'F', 'F']
共生成418个单点突变任务
已生成individual_list.txt


## 4.分析vina结果筛选

In [ ]:
import os
import re
import math
import glob
import argparse
import warnings
from collections import defaultdict
import pandas as pd

# ligand2中C24 C25 环O在vina输出的pose中的ATOM ID(需要在pymol中比对一下)
EPOXIDE_C24 = 29
EPOXIDE_O = 30
EPOXIDE_C25 = 31

# 核心催化原子:ASP101-OD1-OD2 150TYR-OH 230TYR-OH
CATALYTIC_ATOMS = {
    "D101_OD1": (101, "OD1"),
    "D101_OD2": (101, "OD2"),
    "Y150_OH":  (150, "OH"),
    "Y230_OH":  (230, "OH"),
}

# 用来过滤，未结合状态下的结合能
VINA_REF_STATE = 1.981

'''
功能:处理每个protein的pdbqt文件的[[每行]]字段
'''
def _parse_atom_line(ln):
    try:
        serial = int(ln[6:11].strip()) # 原子序列号
        name = ln[12:16].strip() # 原子名字
        resn = ln[17:20].strip() # 残基名字
        chain = ln[21].strip() # 链
        resnum = int(ln[22:26].strip()) # 残基编号
        # 原子坐标
        x = float(ln[30:38].strip())
        y = float(ln[38:46].strip())
        z = float(ln[46:54].strip())
        return {
            "serial": serial, "name": name, "resn": resn,
            "chain": chain, "resnum": resnum,
            "x": x, "y": y, "z": z,
        }
    except (ValueError, IndexError):
        return None

'''
功能:找到每个MUT的pdbqt对应的CATALYTIC_ATOMS核心催化原子坐标
    {'D101_OD1': (-1.08, 4.646, -1.312),
     'D101_OD2': (-2.657, 5.832, -0.331),
     'Y150_OH': (-4.356, 9.937, -3.137),
     'Y230_OH': (-1.381, 7.352, -4.973)}
'''
def parse_receptor_catalytic(pdbqt_path):
    # 这里存放每行的原子对应坐标的(resnum, name):(x, y, z)
    found = {}
    with open(pdbqt_path) as fh:
        for ln in fh:
            if not ln.startswith(("ATOM", "HETATM")):
                continue
            a = _parse_atom_line(ln)
            if a is None:
                continue
            key = (a["resnum"], a["name"])
            if key not in found:
                found[key] = (a["x"], a["y"], a["z"])
    coords = {}
    # 遍历我们想要的原子坐标进行存放到coords中
    for col, (resnum, aname) in CATALYTIC_ATOMS.items():
        coords[col] = found.get((resnum, aname))
    return coords

'''
功能:计算ligad输出的pose中的每个model的 第几个pose | 结合能 | rmsd_lb | rmsd_ub | 每个原子的信息
'''
def parse_ligand_models(pdbqt_path):
    text = open(pdbqt_path).read()
    # 把每个pose的结果分开
    '''
    ('1',
     '\nREMARK VINA RESULT:      -7.5      0.000 ...')
    '''
    model_blocks = re.findall(r"^MODEL\s+(\d+)(.*?)^ENDMDL", text, re.S | re.M)
    models = []
    for mnum_str, block in model_blocks:
        mnum = int(mnum_str)
        affinity = rmsd_lb = rmsd_ub = None
        # 拿到结合能
        for rln in block.splitlines():
            if not rln.startswith("REMARK VINA RESULT"):
                continue
            parts = rln.split()
            # parts: ['REMARK', 'VINA', 'RESULT:', aff, rmsd_lb, rmsd_ub]
            # ['REMARK', 'VINA', 'RESULT:', '-7.5', '0.000', '0.000']
            # ['REMARK', 'VINA', 'RESULT:', '1.981', '2.428', '8.384']
            # 两行，取第一行，过滤第二行
            if len(parts) < 6:
                continue
            try:
                aff = float(parts[3])
                rlb = float(parts[4])
                rub = float(parts[5])
            except ValueError:
                continue
            # 配体未结合态的结合能1.981
            if abs(rlb - VINA_REF_STATE) < 1e-3:
                continue
            affinity, rmsd_lb, rmsd_ub = aff, rlb, rub
            # 拿到第一个REMARK VINA RESULT 的结果就是pose的结果直接终止
            break

        # 取pose的所有ATOM行
        atoms_by_serial = {}
        for ln in block.splitlines():
            if not ln.startswith(("ATOM", "HETATM")):
                continue
            a = _parse_atom_line(ln)
            if a is not None:
                atoms_by_serial[a["serial"]] = a

        models.append({
            "model": mnum, # 第几个pose
            "affinity": affinity, # 结合能
            "rmsd_lb": rmsd_lb,
            "rmsd_ub": rmsd_ub,
            "atoms": atoms_by_serial, # 每个pose的所有ATOM信息，最重要的坐标
        })
    return models

'''
功能:计算原子之间的欧式距离
'''
def dist(a, b):
    if a is None or b is None:
        return None
    return math.sqrt(sum((x - y) ** 2 for x, y in zip(a, b)))

'''
功能:计算原子之间的角度
    C24或C25为角度
'''
def angle_deg(p1, p2, p3):
    if p1 is None or p2 is None or p3 is None:
        return None
    v1 = [p1[i] - p2[i] for i in range(3)]
    v2 = [p3[i] - p2[i] for i in range(3)]
    n1 = math.sqrt(sum(c * c for c in v1))
    n2 = math.sqrt(sum(c * c for c in v2))
    if n1 == 0 or n2 == 0:
        return None
    cosv = max(-1.0, min(1.0, sum(v1[i] * v2[i] for i in range(3)) / (n1 * n2)))
    return math.degrees(math.acos(cosv))

'''
功能:拿到ligand中的pose的ATOM的需要计算原子坐标
'''
def _coord(atom_dict):
    if atom_dict is None:
        return None
    return (atom_dict["x"], atom_dict["y"], atom_dict["z"])

'''
功能:计算需要的几何信息
'''
def analyze_mutant(mutant_name, receptor_pdbqt, out_pdbqt):
    # 拿到MUT需要的原子坐标
    cat = parse_receptor_catalytic(receptor_pdbqt)
    # 拿到ligand每个pose的信息
    models = parse_ligand_models(out_pdbqt)
    # 
    missing_cat = [k for k, v in cat.items() if v is None]
    if missing_cat:
        warnings.warn(
            f"[{mutant_name}]丢失催化原子: {missing_cat}"
        )
    rows = []
    for m in models:
        # 拿到ligand中的C24 O C25三个的坐标
        lig = m["atoms"]
        c24 = _coord(lig.get(EPOXIDE_C24))
        oep = _coord(lig.get(EPOXIDE_O))
        c25 = _coord(lig.get(EPOXIDE_C25))
        # 拿到MUT的4个原子坐标
        od1 = cat["D101_OD1"]
        od2 = cat["D101_OD2"]
        y150 = cat["Y150_OH"]
        y230 = cat["Y230_OH"]

        # 计算几何距离和角度
        d_od1_c24 = dist(od1, c24)
        d_od2_c24 = dist(od2, c24)
        d_od1_c25 = dist(od1, c25)
        d_od2_c25 = dist(od2, c25)
        a_od1_c24 = angle_deg(od1, c24, oep) # OD1 - C24 - O
        a_od2_c24 = angle_deg(od2, c24, oep) # OD2 - C24 - O
        a_od1_c25 = angle_deg(od1, c25, oep) # OD1 - C25 - O
        a_od2_c25 = angle_deg(od2, c25, oep) # OD2 - C25 - O
        d_y150 = dist(y150, oep)
        d_y230 = dist(y230, oep)

        rows.append({
            "mutant": mutant_name,
            "rank": m["model"],
            "affinity": m["affinity"],
            "rmsd_lb": m["rmsd_lb"],
            "rmsd_ub": m["rmsd_ub"],
            "d_OD1_C24": round(d_od1_c24, 3) if d_od1_c24 is not None else None,
            "d_OD2_C24": round(d_od2_c24, 3) if d_od2_c24 is not None else None,
            "angle_OD1_C24": round(a_od1_c24, 1) if a_od1_c24 is not None else None,
            "angle_OD2_C24": round(a_od2_c24, 1) if a_od2_c24 is not None else None,
            "d_OD1_C25": round(d_od1_c25, 3) if d_od1_c25 is not None else None,
            "d_OD2_C25": round(d_od2_c25, 3) if d_od2_c25 is not None else None,
            "angle_OD1_C25": round(a_od1_c25, 1) if a_od1_c25 is not None else None,
            "angle_OD2_C25": round(a_od2_c25, 1) if a_od2_c25 is not None else None,
            "d_Y150_Oepox": round(d_y150, 3) if d_y150 is not None else None,
            "d_Y230_Oepox": round(d_y230, 3) if d_y230 is not None else None,
        })
    return rows

'''
功能:找到MUTs以及对应的_out文件路径，同时保存没有匹配的
    也可以不用这步，核对一下最好
'''
def discover_mutants(receptor_dir, out_dir):
    rec_files = glob.glob(os.path.join(receptor_dir, "*.pdbqt"))
    out_files = glob.glob(os.path.join(out_dir, "*_out.pdbqt"))
    # 键:basename(EPH_relaxed_0001_pdb2pqr_Repair_100_r2_0001) 
    # 值:path(10_MUTs_foldx_pdb2pqr_meeko\\EPH_relaxed_0001_pdb2pqr_Repair_100_r2_0001.pdbqt)
    rec_map = {}
    for p in rec_files:
        # EPH_relaxed_0001_pdb2pqr_Repair_100_r2_0001
        # basename=EPH_relaxed_0001_pdb2pqr_Repair_100_r2_0001.pdbqt
        # splitext拆分:('EPH_relaxed_0001_pdb2pqr_Repair_100_r2_0001', '.pdbqt')
        base = os.path.splitext(os.path.basename(p))[0]
        rec_map[base] = p
    # {'EPH_relaxed_0001_pdb2pqr_Repair_100_r2_0001': '11_vina_results\\EPH_relaxed_0001_pdb2pqr_Repair_100_r2_0001_out.pdbqt'}
    out_map = {}
    for p in out_files:
        base = os.path.splitext(os.path.basename(p))[0]
        if base.endswith("_out"):
            # 结果中去掉_out
            base = base[:-4]
        out_map[base] = p
    '''
    out_map和rec_map中key是一样的,方便比对
    '''
    matched = []
    unmatched_rec = []
    for base in sorted(rec_map):
        if base in out_map:
            # MUTs以及对应的vina  _out结果路径
            matched.append((base, rec_map[base], out_map[base]))
        else:
            unmatched_rec.append(base)

    unmatched_out = [b for b in sorted(out_map) if b not in rec_map]
    return matched, unmatched_rec, unmatched_out

'''
功能:导出结果
'''
def run_analysis(receptor_dir, out_dir, output="vina_result.csv"):
    # 先找MUTs和_out结果文件
    matched, unmatched_rec, unmatched_out = discover_mutants(receptor_dir, out_dir)

    # 运行前的文件检查
    print(f"找到{len(matched)}个蛋白配体对")
    if unmatched_rec:
        print(f"WARNING: {len(unmatched_rec)}个蛋白没有对应的vina结果")
        for b in unmatched_rec[:10]:
            print(f"{b}")
        if len(unmatched_rec) > 10:
            print(f"还有{len(unmatched_rec) - 10}更多")
    if unmatched_out:
        print(f"WARNING: {len(unmatched_out)} 有结果没有对应的蛋白")
        for b in unmatched_out[:10]:
            print(f"{b}")
        if len(unmatched_out) > 10:
            print(f"还有{len(unmatched_out) - 10}更多")

    if not matched:
        print("ERROR: 没有蛋白配体对，请检查文件")
        return

    all_rows = []
    total_poses = 0
    skipped = 0

    for i, (mutant_name, rec_path, out_path) in enumerate(matched, 1):
        try:
            rows = analyze_mutant(mutant_name, rec_path, out_path)
            all_rows.extend(rows)
            total_poses += len(rows)
            print(f"[{i}/{len(matched)}] {mutant_name}: {len(rows)}poses")
        except Exception as e:
            skipped += 1
            print(f"[{i}/{len(matched)}] {mutant_name}: ERROR - {e}")
            continue

    df = pd.DataFrame(all_rows)

    # 验证列
    col_order = [
        "mutant", "rank", "affinity", "rmsd_lb", "rmsd_ub",
        "d_OD1_C24", "d_OD2_C24", "angle_OD1_C24", "angle_OD2_C24",
        "d_OD1_C25", "d_OD2_C25", "angle_OD1_C25", "angle_OD2_C25",
        "d_Y150_Oepox", "d_Y230_Oepox",
    ]
    df = df[[c for c in col_order if c in df.columns]]

    df.to_csv(output, index=False)
    print(f"结束:{len(matched) - skipped}/{len(matched)}"
            f"共{total_poses}poses, {skipped}跳过.")
    print(f"输出csv文件:{output}")
    return df

'''
主函数:参数设置和调用，这里是给py文件调用用的
    python *.py \
        --receptor-dir /receptor_dir \
        --out-dir /out_dir \
        --output vina_result.csv
'''
def main():
    ap = argparse.ArgumentParser(description="批量处理vina对接结果")
    ap.add_argument("--receptor-dir", required=True, help="蛋白文件夹路径")
    ap.add_argument("--out-dir", required=True, help="vina对接结果文件路径")
    ap.add_argument("--output", default="vina_result.csv", help="最终输出的csv分析文件")
    args = ap.parse_args()
    run_analysis(args.receptor_dir,args.out_dir,args.output)

'''
调用运行
'''
# 测试用例
# pdbqtpath = "./10_MUTs_foldx_pdb2pqr_meeko"
# ligandpath = "./11_vina_results"
# outputfile = "test_vina_result.csv"
# 实际跑结果
pdbqtpath = "./10_MUTs_foldx_pdb2pqr_meeko"
ligandpath = "./11_vina_results"
outputfile = "vina_result.csv"
df = run_analysis(pdbqtpath, ligandpath, outputfile)


usage: ipykernel_launcher.py [-h] --receptor-dir RECEPTOR_DIR --out-dir
                             OUT_DIR [--output OUTPUT]
ipykernel_launcher.py: error: the following arguments are required: --receptor-dir, --out-dir


SystemExit: 2

d:\Anconda\envs\pytorch_env\lib\site-packages\IPython\core\interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
